In [0]:
dbutils.widgets.text("source_system", "")
dbutils.widgets.text("table_name", "")
dbutils.widgets.text("landing_timestamp", "")
dbutils.widgets.text("source_file_format", "")

source_system = dbutils.widgets.get("source_system")
table_name = dbutils.widgets.get("table_name")
landing_timestamp = dbutils.widgets.get("landing_timestamp")
source_file_format = dbutils.widgets.get("source_file_format")


In [0]:
import os

landing_path = f"/Volumes/caresync_databricks/landing/landing_volume/{source_system}/{table_name}/{landing_timestamp}/"

if not os.path.exists(landing_path):
    print(f"No new files received at path: {landing_path}")
    dbutils.notebook.exit("No new files received")

In [0]:
if source_file_format == "csv":
    landing_df = spark.read.format("csv").option("header", True).option("inferSchema", True).load(landing_path)
else:
    landing_df = spark.read.format("parquet").load(landing_path)

In [0]:
from pyspark.sql import functions as F

updated_df = landing_df.withColumn("landing_timestamp", F.lit(landing_timestamp)) \
    .withColumn("insert_timestamp", F.current_timestamp())


In [0]:
updated_df.write.mode("append").saveAsTable(f"caresync_databricks.bronze.{table_name}") 